## Core of the Transformer Model


for every head- Q,K,V -> dimension =64 

8 heads

combined dimension =64*8 =512

In [ ]:
"""
TRANSFORMER TEXT GENERATION - REORGANIZED FOR LEARNING
------------------------------------------------------------------------------
Better structure: Overview → Data → Architecture → Training → Generation
Preserving all the detailed understanding comments!
"""

import os
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.layers import Layer, Embedding, Dense, LayerNormalization, Dropout
import numpy as np


# SECTION 1: CONFIGURATION & HYPERPARAMETERS
# -------------------------------------------------------------------------------
# Put all important settings at the TOP so they're easy to find and modify

class Config:
    """All hyperparameters in one place - easy to experiment!"""
    # Data settings
    FILE_PATH = "../../data/harry_potter.txt"
    SEQ_LENGTH = 100  # context window size --> Means The model can only "remember" 100 previous tokens
    
    # Model architecture
    EMBED_DIM = 128      # Each token (word) becomes a vector of length 128. -> "harry" → [0.12, -0.98, 0.44, ..., 0.03]  (128 numbers)
                         # Small model → 64–256  GPT-3 → 12288 (very large)
    NUM_HEADS = 4        # Number of attention heads, Of course its multi-head attention and not single head attention 
                            # Example intuition:
                            # Head 1 → grammar
                            # Head 2 → long-range dependency
                            # Head 3 → entities
                            # Head 4 → local context
                            
                         # ✨each head will have (embed_dim / num_heads) dimensions ie. 128/4 = 32 dimensions
                         # ✨But good news is heads are concatenated back to embed_dim after attention i.e. 128
    FF_DIM = 512         # Feed-forward layer size , This is the hidden size of the feed-forward network inside each Transformer block.
                         # Think of it as: “After reading, how deeply do I think about each word?”
                         # Rule of thumb (IMPORTANT):ff_dim ≈ 4 × embed_dim
                         
    NUM_LAYERS = 1       # 🌟🌟 Number of Transformer layers (depth of the model)
                         # you can change this to 2, 4, 6, 8, etc. 🌟🌟
                         #  1 layer  → shallow thought
                         #  6 layers → structured reasoning
                         #  12 layers → deep abstraction
                         
    DROPOUT_RATE = 0.1   # Regularization
    
    """
        One Mental Model (remember this):

        embed_dim → what a word knows
        num_heads → how many ways it looks
        ff_dim → how deeply it thinks
        num_layers → how many times it thinks
        
    """
    
    # Training settings
    EPOCHS = 50
    BATCH_SIZE = 128
    VALIDATION_SPLIT = 0.1
    
    # Generation settings
    MODEL_SAVE_PATH = "../../models/harry_transformer_model.keras"


config = Config()






STEP 1: LOADING DATA
Dataset length: 457729 characters
Vocabulary size: 6663 unique words
input_sequences: 80922
 Data shape: X=(80922, 100), y=(80922,)

STEP 2: BUILDING MODEL
After embeddings: (None, 100, 128)
After transformer block 1: (None, 100, 128)
After taking last token: (None, 128)
Final output: (None, 6663)

✅ Model built successfully!
Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 100)]             0         
                                                                 
 token_and_position_embeddin  (None, 100, 128)         865664    
 g_3 (TokenAndPositionEmbedd                                     
 ing)                                                            
                                                                 
 transformer_block_3 (Transf  (None, 100, 128)         198272    
 ormerBlock)                           

In [ ]:
# SECTION 2: DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
# Load data first, then we'll understand what we're modeling

def load_data(file_path):
    """Load Harry Potter book as the dataset
    url -> https://www.kaggle.com/datasets/shubhamindola/harry-potter-books
    """
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    return text


def prepare_sequences(text, seq_length):
    """
    Convert text into training sequences.
    
    First seq_length tokens (input): Used for training the model.
    Last token (target): Used as the label the model tries to predict.
    So total of (500 + 1) in one input_sequence index
    
    Returns:
        X: Input sequences (batch, seq_length)
        y: Target words (batch,)
        tokenizer: For converting text ↔ numbers
        total_words: Vocabulary size (below total_words = 6662 - basically all tokens in the text)
    """
    # Tokenize the text
    tokenizer = Tokenizer(oov_token="<OOV>")
    tokenizer.fit_on_texts([text])
    total_words = len(tokenizer.word_index) + 1
    
    print(f"Dataset length: {len(text)} characters")
    print(f"Vocabulary size: {total_words} unique words")
    
    # Convert text to sequences
    # expected list of texts, so we pass [text] and get first element of the output
    tokens = tokenizer.texts_to_sequences([text])[0]
    
    # Create input sequences with sliding window
    input_sequences = []
    for i in range(seq_length, len(tokens)):
        input_sequences.append(tokens[i - seq_length : i + 1])
    
    print(f"input_sequences: {len(input_sequences)}")
    
    # Pad sequences and split inputs/targets
    # After this X will have inputs and y will have label for those inputs
    input_sequences = np.array(
        pad_sequences(input_sequences, maxlen=seq_length + 1, padding="pre")
    )
    
    X, y = input_sequences[:, :-1], input_sequences[:, -1]
    
    # Note: there are other ways for encoding like pre-trained word2vec encoding and so on
    # y = tf.keras.utils.to_categorical(y, num_classes=total_words)  # One-hot encoding alternative
    
    return X, y, tokenizer, total_words


# Load and prepare data
print("\n" + "="*70)
print("STEP 1: LOADING DATA")
print("="*70)
text = load_data(config.FILE_PATH).lower()
X, y, tokenizer, total_words = prepare_sequences(text, config.SEQ_LENGTH)
print(f" Data shape: X={X.shape}, y={y.shape}")





In [ ]:
# SECTION 3: CUSTOM LAYERS (BUILDING BLOCKS)
# --------------------------------------------------------------------------------
# Now define the architecture components with detailed explanations

class TokenAndPositionEmbedding(Layer):
    """
    Combines word embeddings + positional embeddings.
    
    The Embedding layer takes an integer tensor and replaces
    each integer with an embed_dim-sized vector
    
    Example — positions = [0, 1, 2, 3]
    After embedding — positions = [
      [0.2, 0.1, 0.3, 0.5, 0.6, 0.9, 0.7, 0.8],  # Position 0
      [0.4, 0.2, 0.1, 0.6, 0.5, 0.7, 0.9, 0.3],  # Position 1
      [0.5, 0.3, 0.8, 0.2, 0.7, 0.4, 0.6, 0.1],  # Position 2
      [0.9, 0.6, 0.2, 0.3, 0.1, 0.8, 0.4, 0.7],  # Position 3
    ]
    
    Initial shape of x → (batch_size, seq_len)
    batch_size → number of sentences in a batch
    seq_len → number of tokens (words) in each sentence
    each value in x is an integer index from 0 to vocab_size - 1
    After embedding → (batch_size, seq_len, embed_dim)
    
    Example — embed_dim = 8, batch_size = 2
    x = [
      [
        [0.2, 0.1, 0.4, 0.3, 0.8, 0.7, 0.6, 0.9],  # Token 2
        [0.5, 0.3, 0.9, 0.1, 0.2, 0.6, 0.8, 0.7],  # Token 5
        [0.4, 0.9, 0.2, 0.3, 0.1, 0.7, 0.5, 0.6],  # Token 1
        [0.3, 0.8, 0.6, 0.2, 0.5, 0.9, 0.7, 0.4],  # Token 7
      ],  # First sentence
     
      [
        [0.1, 0.6, 0.9, 0.7, 0.3, 0.5, 0.2, 0.8],  # Token 0
        [0.4, 0.2, 0.3, 0.9, 0.7, 0.5, 0.1, 0.6],  # Token 3
        [0.8, 0.5, 0.4, 0.1, 0.6, 0.3, 0.2, 0.7],  # Token 8
        [0.9, 0.3, 0.5, 0.7, 0.8, 0.2, 0.6, 0.1],  # Token 4
      ],  # Second sentence
    ]
    """
    
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super(TokenAndPositionEmbedding, self).__init__(**kwargs)
        self.maxlen = maxlen
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        
        # Word embeddings: vocab_size → embed_dim
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        
        # Position embeddings: maxlen → embed_dim
        self.pos_emb = Embedding(input_dim=maxlen, output_dim=embed_dim)
    
    def call(self, x):
        # The maximum sequence length the model can handle
        maxlen = tf.shape(x)[-1]  # sets maxlen to the length of the input sequence
        # -1 refers to the last dimension of x, which is seq_len
        
        # Generate [0, 1, 2, ..., maxlen-1]
        positions = tf.range(start=0, limit=maxlen, delta=1) # delta=1 means step size of 1
        
        # Each position index is mapped to a trainable embedding of shape (maxlen, embed_dim)
        positions = self.pos_emb(positions)
        
        # Each token ID in x is mapped to an embedding of shape (batch_size, maxlen, embed_dim)
        x = self.token_emb(x)
        
        # x has shape (batch_size, seq_len, embed_dim)
        # positions has shape (maxlen, embed_dim)
        # But maxlen == seq_len, so positions effectively has shape (seq_len, embed_dim)
        # TensorFlow broadcasts positions across batch_size, treating it as if it were (1, seq_len, embed_dim)
        # This allows element-wise addition between x and position
        return x + positions
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "maxlen": self.maxlen,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim,
        })
        return config


class MultiHeadAttention(Layer):
    """
    The CORE of the Transformer: Multi-Head Self-Attention
    
    embed_dim = dimension of Q, K, and V before splitting into multiple heads
    It is same as total dimension of the input embeddings (word embeddings)
    Example: embed_dim = 512
    
    num_heads = number of attention heads
    Example: num_heads = 4
    
    projection_dim = Size of each attention head's subspace
    Each head gets a smaller subspace of the embedding dimension
    projection_dim = embed_dim // num_heads
    Example: projection_dim = 512 // 4 = 128
    
    Fully connected (dense) layers that project the input into Q, K, V
    These layers map the input embeddings to the same embed_dim
    These layers will be reshaped / split later to split across attention heads
    A single large matrix multiplication is more efficient than many small ones
    GPUs love large matrix multiplications because they are optimized for parallel computation
    This allows TF/Keras to efficiently batch the computation, leveraging better GPU memory utilization
    
    Q determines "what to focus on"
    K acts as "labels" to be matched with queries
    V holds the actual information
    """
    
    def __init__(self, embed_dim, num_heads, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads  # example = 4
        self.embed_dim = embed_dim  # example = 128
        self.projection_dim = embed_dim // num_heads  # example = 32
        
        # Dense layers for Q, K, V projections
        self.query_dense = Dense(embed_dim)
        self.key_dense = Dense(embed_dim)
        self.value_dense = Dense(embed_dim)
        
        # After multi-head attention is applied, the outputs from all heads
        # are concatenated back into embed_dim
        self.combine_heads = Dense(embed_dim)
    
    def attention(self, query, key, value):
        """
        Scaled Dot-Product Attention
        """
        scores = tf.matmul(query, key, transpose_b=True)  # Q * K^T  ,This gives us the raw attention scores
        
        # Scaling factor (convert int to float32)
        scores /= tf.math.sqrt(tf.cast(self.projection_dim, scores.dtype)) # scores.dtype ensures no type mismatch, This gives us more stable gradients
        
        # Softmax along last axis so rows sum to 1 (probability along rows)
        attention_probs = tf.nn.softmax(scores, axis=-1) # This gives us the attention weights
        
        # The higher the score, the more focus that token gets
        # Softmax should be applied along the keys
        # (i.e., across the last dimension of the scores matrix)
        # Each row corresponds to a query token attending to all key tokens
        # This ensures that each query distributes its attention across all keys properly
        # Each row sums to 1
        
        return tf.matmul(attention_probs, value), attention_probs  # Probabilities * V
    
    def split_heads(self, x, batch_size):
        """
        x = query, key or value with shape → (batch_size, seq_len, embed_dim)
        batch_size = number of sequences being processed in parallel (for batch processing)
        
        x->taking input and splitting into multiple heads
        when we don't have fixed number of words we use -1 (seq_len)
        
        New shape → batch_size, num of words (seq_len), num_heads, projection
        Shape we want → batch_size, num_heads, num of words (seq_len), projection
        ✨ Adla badli closely dekho of num_heads and num of words
        Before concatenation we need to have num_heads before num of words so we transpose it
        batch_size of (8 heads of (4 words * 64 dimension))
        
        Before transpose  → (batch_size, seq_len, num_heads, projection_dim)
        After transpose   → (batch_size, num_heads, seq_len, projection_dim)
        ✨ Adla badli by transposing
        
        The -1 in tf.reshape is a placeholder that tells TensorFlow to automatically
        infer that dimension's value based on the total number of elements in the tensor
        -1 is replaced by seq_len by TensorFlow
        
        ✨tf.reshape(tensor, shape)
        ✨x.shape = (batch_size, seq_len, embed_dim)
        """
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim)) # reshape to split heads
        # attention computation expects shape → (batch_size, num_heads, seq_len, projection_dim)
        return tf.transpose(x, perm=[0, 2, 1, 3]) # Reorder dimensions to put num_heads before seq_len
    
    def call(self, inputs):
        """
        In TF/Keras — call(self, inputs) is a standard method used inside Layer subclasses
        to define the forward pass of a neural network layer
        
        Whenever layer is called this function is executed
        This is starter function for custom layers
        
        Earlier inputs shape → batch_size, num of words (seq_len), embed_dim
        """
        query, key, value = inputs
        batch_size = tf.shape(query)[0]
        
        # Project and split into heads
        query = self.split_heads(self.query_dense(query), batch_size)
        key = self.split_heads(self.key_dense(key), batch_size)
        value = self.split_heads(self.value_dense(value), batch_size)
        
        # Apply attention
        attention, _ = self.attention(query, key, value)
        
        # Shape I have → batch_size, num_heads, num of words (seq_len), projection_dim  ✨ before transpose
        # Shape I want → batch_size, num of words (seq_len), num_heads, projection_dim  ✨ after transpose
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        
        # Concatenate heads
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim)) #✨ back to old shape i,e, (batch_size, seq_len, embed_dim)
        
        return self.combine_heads(concat_attention)
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
        })
        return config

class TransformerBlock(Layer):
    """
    Complete Transformer layer combining:
    1. Multi-Head Attention
    2. Feed-Forward Network
    3. Residual connections
    4. Layer normalization
    5. Dropout
    """
    
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate
        
        self.att = MultiHeadAttention(embed_dim, num_heads)
        
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        
        # y = (x - mean) / sqrt(variance + epsilon)
        # epsilon ensures we never divide by zero
        # it is small enough not to affect the result but large enough to prevent instability
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)
    
    def call(self, inputs, training):
        attn_output = self.att((inputs, inputs, inputs))
        
        # Dropout randomly deactivates some neurons during training to reduce overfitting
        # Ensure dropout is only applied during training, not inference
        attn_output = self.dropout1(attn_output, training=training)
        
        out1 = self.layernorm1(inputs + attn_output)  # Residual Connection
        
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        
        return self.layernorm2(out1 + ffn_output)  # Residual Connection
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "rate": self.rate,
        })
        return config



# SECTION 4: BUILD THE MODEL
# ---------------------------------------------------------------------------------------
# Now assemble everything into a complete model

print("\n" + "="*70)
print("STEP 2: BUILDING MODEL")
print("="*70)

def build_transformer_model(maxlen, vocab_size, embed_dim, num_heads, ff_dim, num_layers=1):
    """
    Build the complete Transformer model.
    
    Architecture:
    Input (word IDs) 
      → Token + Position Embeddings 
      → Transformer Block(s) 
      → Take last token 
      → Dense (predict next word)
    """
    
    # Input layer
    inputs = tf.keras.Input(shape=(maxlen,))
    
    # Embeddings
    embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
    x = embedding_layer(inputs)
    print(f"After embeddings: {x.shape}")
    
    # Stack multiple transformer blocks
    # 🌟🌟 Number of Transformer layers (depth of the model)
    # Instead of a single TransformerBlock, define layers explicitly:
    # num_layers = 6  # you can change this to 2, 4, 8, etc.
    # 
    # for _ in range(num_layers):
    #     x = TransformerBlock(
    #         embed_dim=embed_dim,
    #         num_heads=num_heads,
    #         ff_dim=ff_dim
    #     )(x) 🌟🌟
    
    for i in range(num_layers):
        transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
        x = transformer_block(x)
        # x = transformer_block(x, training=True)  # Can explicitly set training mode
        print(f"After transformer block {i+1}: {x.shape}")
    
    # Take only the last token's representation
    # For next-word prediction, we only need the final output
    # The last position contains information about the entire sequence due to attention
    x = x[:, -1, :]
    print(f"After taking last token: {x.shape}")
    
    # Output layer: probability distribution over vocabulary
    x = Dense(vocab_size, activation="softmax")(x)
    print(f"Final output: {x.shape}")
    
    return tf.keras.Model(inputs=inputs, outputs=x)


# Build model
model = build_transformer_model(
    maxlen=config.SEQ_LENGTH,
    vocab_size=total_words,
    embed_dim=config.EMBED_DIM,
    num_heads=config.NUM_HEADS,
    ff_dim=config.FF_DIM,
    num_layers=config.NUM_LAYERS
)

print(f"\n✅ Model built successfully!")
model.summary()



# SECTION 5: COMPILE & TRAIN
# ---------------------------------------------------------------------------

print("\n" + "="*70)
print("STEP 3: TRAINING MODEL")
print("="*70)

# Compile the model
# model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])  # For one-hot labels
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",  # For integer labels
    metrics=["accuracy"]
)
# metrics = ["perplexity"] means how much your model is confused while predicting the next word
# lower the perplexity better the model and vice versa

# Train the model
history = model.fit(
    X, y,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    validation_split=config.VALIDATION_SPLIT,
    verbose=1
)

# Save model
os.makedirs("models", exist_ok=True)
model.save(config.MODEL_SAVE_PATH)
print(f"\n✅ Model saved to {config.MODEL_SAVE_PATH}")



# SECTION 6: TEXT GENERATION
# -----------------------------------------------------------------------

print("\n" + "="*70)
print("STEP 4: GENERATING TEXT")
print("="*70)

# Load model (with custom layers)
custom_objects = {
    "MultiHeadAttention": MultiHeadAttention,
    "TransformerBlock": TransformerBlock,
    "TokenAndPositionEmbedding": TokenAndPositionEmbedding,
}

loaded_model = tf.keras.models.load_model(
    config.MODEL_SAVE_PATH,
    custom_objects=custom_objects
)

print("Model loaded successfully.", loaded_model)


def generate_text(seed_text, next_words, max_sequence_len):
    """
    Generate text by repeatedly predicting the next word.
    
    Process:
    1. Start with seed text
    2. Convert to tokens → pad → predict
    3. Take word with highest probability
    4. Append to seed and repeat
    """
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len - 1,
            padding="pre"
        )
        predicted = loaded_model.predict(token_list, verbose=0)
        predicted_word = tokenizer.index_word[np.argmax(predicted)]
        seed_text += " " + predicted_word
    
    return seed_text


# Generate text
# seed_text = "harry looked at"
seed_text = "hermione granger was almost as nervous"
generated_text = generate_text(
    seed_text,
    next_words=50,
    max_sequence_len=config.SEQ_LENGTH + 1
)

print(f"\nGenerated text length: {len(generated_text)} characters")
print(f"\nGenerated text:\n{generated_text}")


In [23]:
seed_text = "harry looked at ron"
generated_text = generate_text(
    seed_text,
    next_words=50,
    max_sequence_len=config.SEQ_LENGTH + 1
)

print(f"\nGenerated text length: {len(generated_text)} characters")
print(f"\nGenerated text:\n{generated_text}")  


Generated text length: 269 characters

Generated text:
harry looked at ron â€œyou donâ€™t think so was no one ever remembered that the last word harry had been watching the hat had bowed in case for the last shop on the other side three of them â€œmiss granger you foolish package just read by a bit of a week â€ said harry


## What Is Missing Compared to ChatGPT?

- **Masked Attention**  
  ChatGPT uses causal masking so that a word cannot see future words during training.  
  Our model uses regular attention, which allows it to see the entire sequence.

- **Multiple Stacked Transformer Blocks**  
  ChatGPT has many layers (e.g., 12, 24, 96 layers).  
  Our model has only one Transformer block.

- **Tokenization & Byte-Pair Encoding (BPE)**  
  ChatGPT does not use simple tokenization; it uses Byte-Pair Encoding (BPE) or WordPiece for better vocabulary handling.  
  Our model uses basic word tokenization.

- **Training on Large Datasets**  
  ChatGPT is trained on hundreds of GBs of text.  
  Our model is trained on a single Harry Potter book (very limited).

- **Decoding Strategies for Text Generation**  
  ChatGPT uses sampling (top-k, nucleus sampling) or beam search to generate text.  
  Our model does not have a decoding strategy.


### Key Transformer Concepts

- **Context window** → how much the model can see  
  (number of tokens available at once for attention)

- **Layers** → how deeply the model can reason  
  (number of Transformer blocks stacked on top of each other)

In simple terms:
- Increasing the **context window** improves memory
- Increasing the **number of layers** improves reasoning depth

Both must be balanced for an effective language model.
